# 05 — Validate the candidate's contract

**Foundry feature:** the piece Foundry's own composite score doesn't give you — deterministic,
rule-level gating of an optimized candidate before you promote it. **Mode: CLI.**

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## First, validate the *un-optimized baseline* — a known input

Before scoring the candidate, run the same tool against the original `instructions.md`. This case
study's baseline has exactly one known, documented flaw (a hedged sign-off line, `should_remove:SR-01`)
— seeing the validator catch precisely that, and nothing else critical, is how you calibrate what its
report format means before trusting it on a real candidate.

In [ ]:
run(["python3", "_tools/validate_candidate.py",
     "--agent", AGENT_ID,
     "--candidate", str(AGENT_DIR / "instructions.md"),
     "--candidate-source", "optimize",
     "--json", "/tmp/baseline_report.json"])

Read the report: `[OK]` lines pass deterministically, `[XX]` lines are deterministic failures
(`should_remove:SR-01` should be one of them — the hedged sign-off is still there in the raw
baseline), and `[??]` lines are `semantic` rules the deterministic pass can't resolve on its own —
notebook 06 handles those.

## Now validate the exported candidate

`--candidate-source optimize` is required and asserts the candidate came from the optimize split,
not holdout — `validate_candidate.py` raises if you pass anything else, because scoring a candidate
that was itself tuned against the holdout set defeats the leakage guard from notebook 03.

In [ ]:
candidate_path = AGENT_DIR / "candidates" / "foundry_run1.md"
assert candidate_path.exists(), "Run notebook 04 first to produce (or place your real export at) this path."

result = run(["python3", "_tools/validate_candidate.py",
              "--agent", AGENT_ID,
              "--candidate", str(candidate_path),
              "--candidate-source", "optimize",
              "--json", str(candidate_path.with_suffix(".report.json"))])

In [ ]:
report = json.loads((candidate_path.with_suffix(".report.json")).read_text())
print("blocked:", report["blocked"])
print("critical failures:", [f["id"] for f in report["critical_failures"]] if report.get("critical_failures") else [])
print("unjudged semantic rules:", len(report.get("unjudged", [])))

A line prefixed `[XX]` under **BLOCKED** is a critical failure — don't promote yet. Lines prefixed
`[??]` are semantic rules waiting on a judge; that's exactly what the next notebook resolves.

## Next

Continue to **`06_resolve_semantic_rules_with_judge.ipynb`** to resolve the `UNJUDGED` queue.